In [1]:
import gymnasium as gym
import numpy as np
import subprocess
import threading
import retro
import os
import wave
# from sbx import DDPG, DQN, PPO, SAC, TD3, TQC, CrossQ
from pathlib import Path
from gymnasium.wrappers import RecordVideo
from sdlarch_rl import make

import numpy as np

class DoneWrapper(gym.Wrapper):
    def __init__(self, env):
        super().__init__(env)
        self.count = 0

        self.env = env

    def step(self, action):
        observation, reward, done, trunk, info = self.env.step(action)

        self.count += 1

        if self.count > 3000:
            done = True
        
        return observation, reward, done, trunk, info

os.makedirs("videos", exist_ok=True)


video_filename = "videos/video-episode-0.mp4"
audio_filename = "videos/game_audio.wav"

audio_data = b""
arate = None
framerate = None

data = {
    "audio_data": audio_data,
    "arate": arate,
    "framerate": framerate,
}

        
env = make(
    "GranTurismo3-Ps2"
)

env = DoneWrapper(env)

env = RecordVideo(
    env,
    video_folder="videos/",
    episode_trigger=lambda x: x == 0,
    name_prefix="video"
)


render_mode = "rgb_array"
# render_mode = "human"


obs = env.reset()
done = False
count = 0

while True:
    env.render()
    action = np.zeros(16, dtype=np.uint8)
    action[0] = 1

    obs, reward, done, _, info = env.step(action)

    data['arate'] = env.unwrapped.em.get_audio_rate()
    data['framerate'] = env.unwrapped.em.get_frame_rate()
    sound = env.unwrapped.em.get_audio()
    data['audio_data'] += sound.tobytes()

    count += 1

    if done:
        break

pygame.quit()
env.close()

print("arate: ", data['arate'])
print("framerate: ", data['framerate'])

# save audio
with wave.open(audio_filename, "wb") as wf:
    wf.setnchannels(2)  # Mono
    wf.setsampwidth(2)  # 16-bit PCM
    wf.setframerate(data['arate'])  # Audio rate
    wf.writeframes(data['audio_data'])

video_prefix = "video.mp4"


ffmpeg_cmd = [
    "ffmpeg", "-y",
    "-r", str(data['framerate']),
    "-i", video_filename,
    "-i", audio_filename,
    # "-vf", "scale=1280:720", # hd resolution
    "-vf", "scale=428:240,setdar=16:9,setsar=1", # snes
    # "-vf", "scale=854:480,setdar=16:9,setsar=1", # 16x9
    "-c:v", "libx265",
    # "-preset", "veryslow",
     "-crf", "10", # quality 10 ~ 50 (10 is better)
    "-c:a", "aac",
    "-b:a", "128k",
    "-ac","2",
    "-strict", "experimental",
    "-shortest",
    "videos/" + video_prefix
]

subprocess.run(ffmpeg_cmd)

print("✅ Finished vídeos")

ModuleNotFoundError: No module named 'retro'